In [68]:
import sqlite3,pandas as pd
con=sqlite3.connect("../data/telco.db")
pd.read_csv("../data/telco_clean_using_py.csv").to_sql("customers",con, if_exists="replace", index=False)
def q(sql):
     return pd.read_sql_query(sql, con)
print(q("select count(*) as total_rows from Customers"))
print(q("select sum(totalcharges) from customers " ))

# withourt index col
print(q("select count(*) as total_rows from Customers").to_string(index=False))

sum=(q("select sum(totalcharges) from customers " ).iloc[0,0])
print(sum)


   total_rows
0        7043
   sum(totalcharges)
0         16056168.7
 total_rows
       7043
16056168.7


In [69]:
import os
import sqlite3
import pandas as pd

os.makedirs("data", exist_ok=True)
con = sqlite3.connect("../data/telco.db")

# Load main dataset
df = pd.read_csv("../data/telco_clean_using_py.csv")
df.to_sql("customers", con, if_exists="replace", index=False)

# Create small lookup table for JOIN practice
contract_lookup = pd.DataFrame({
    "Contract": ["Month-to-month", "One year", "Two year"],
    "contract_months":["1","12","24"],
    "commitment_level": ["Flexible", "Medium", "Long-term"]
})
contract_lookup.to_sql("contract_info", con, if_exists="replace", index=False)

def q(sql):
    return pd.read_sql_query(sql, con)


In [70]:
#customer % of leaving
q1 = """
SELECT 
    COUNT(*) AS total_customers,
    SUM(CASE WHEN Churn = 'Yes' THEN 1 ELSE 0 END) AS churned_customers,
    ROUND(
        100.0 * SUM(CASE WHEN Churn = 'Yes' THEN 1 ELSE 0 END) / COUNT(*), 
        2
    ) AS churn_rate_pct
FROM customers;
"""
print("--- Q1: Overall Churn Rate ---")
print(q(q1), "\n")


--- Q1: Overall Churn Rate ---
   total_customers  churned_customers  churn_rate_pct
0             7043               1869           26.54 



In [107]:
q2 = """
SELECT 
    Contract,
    InternetService,
    PaymentMethod,
    COUNT(*) AS customer_count,
    ROUND(SUM(100.0* CASE WHEN Churn = 'Yes' THEN 1 ELSE 0 END) / COUNT(*), 2) AS churn_rate_pct,
    RANK() OVER (
        ORDER BY (1.0 * SUM(CASE WHEN Churn = 'Yes' THEN 1 ELSE 0 END) / COUNT(*)) DESC
    ) AS churn_rank
FROM customers
GROUP BY Contract, InternetService, PaymentMethod
HAVING COUNT(*) >= 50
   AND (1.0 * SUM(CASE WHEN Churn = 'Yes' THEN 1 ELSE 0 END) / COUNT(*)) > (
       SELECT 1.0 * SUM(CASE WHEN Churn = 'Yes' THEN 1 ELSE 0 END) / COUNT(*) 
       FROM customers
   );
"""
print("--- Q2: Top Churning Segments (Above Overall Average) ---")
print(q(q2))


--- Q2: Top Churning Segments (Above Overall Average) ---
         Contract InternetService              PaymentMethod  customer_count  \
0  Month-to-month     Fiber optic           Electronic check            1307   
1  Month-to-month     Fiber optic               Mailed check             201   
2  Month-to-month     Fiber optic  Bank transfer (automatic)             327   
3  Month-to-month     Fiber optic    Credit card (automatic)             293   
4  Month-to-month             DSL           Electronic check             474   
5  Month-to-month             DSL               Mailed check             367   
6  Month-to-month             DSL    Credit card (automatic)             185   

   churn_rate_pct  churn_rank  
0           60.37           1  
1           50.75           2  
2           45.57           3  
3           41.64           4  
4           40.51           5  
5           30.79           6  
6           27.03           7  


In [72]:
# are Month-Month customer likely to leave
q3 = """
SELECT 
    Contract,
    COUNT(*) AS total_customers,
    ROUND(100.0 * SUM(CASE WHEN Churn = 'Yes' THEN 1 ELSE 0 END) / COUNT(*), 2) AS contract_churn_pct,
    (
        SELECT ROUND(100.0 * SUM(CASE WHEN Churn = 'Yes' THEN 1 ELSE 0 END) / COUNT(*), 2)
        FROM customers
    ) AS overall_churn_pct,
    ROUND(
        100.0 * SUM(CASE WHEN Churn = 'Yes' THEN 1 ELSE 0 END) / COUNT(*) - 
        (SELECT 100.0 * SUM(CASE WHEN Churn = 'Yes' THEN 1 ELSE 0 END) / COUNT(*) FROM customers),
        2
    ) AS diff_from_overall_pct
FROM customers
GROUP BY Contract
ORDER BY contract_churn_pct DESC;
"""
print("--- Q3: Contract Churn vs Overall Rate ---")
print(q(q3), "\n")


--- Q3: Contract Churn vs Overall Rate ---
         Contract  total_customers  contract_churn_pct  overall_churn_pct  \
0  Month-to-month             3875               42.71              26.54   
1        One year             1473               11.27              26.54   
2        Two year             1695                2.83              26.54   

   diff_from_overall_pct  
0                  16.17  
1                 -15.27  
2                 -23.71   



In [73]:
#does tenure affect churn
q4 = """
WITH tenure_groups AS (
    SELECT 
        customerID,
        Churn,
        CASE 
            WHEN tenure <= 12 THEN '1. 0-12m'
            WHEN tenure <= 24 THEN '2. 13-24m'
            WHEN tenure <= 48 THEN '3. 25-48m'
            WHEN tenure <= 60 THEN '4. 49-60m'
            ELSE '5. 61-72m'
        END AS tenure_band
    FROM customers
)
SELECT 
    tenure_band,
    COUNT(*) AS total_customers,
    SUM(CASE WHEN Churn = 'Yes' THEN 1 ELSE 0 END) AS churned_customers,
    ROUND(100.0 * SUM(CASE WHEN Churn = 'Yes' THEN 1 ELSE 0 END) / COUNT(*), 2) AS churn_rate_pct,
    SUM(SUM(CASE WHEN Churn = 'Yes' THEN 1 ELSE 0 END)) OVER (
        ORDER BY tenure_band
    ) AS cumulative_churned_customers
FROM tenure_groups
GROUP BY tenure_band
ORDER BY tenure_band;
"""
print("--- Q4: Churn by Tenure Band & Running Total ---")
print(q(q4), "\n")


--- Q4: Churn by Tenure Band & Running Total ---
  tenure_band  total_customers  churned_customers  churn_rate_pct  \
0    1. 0-12m             2186               1037           47.44   
1   2. 13-24m             1024                294           28.71   
2   3. 25-48m             1594                325           20.39   
3   4. 49-60m              832                120           14.42   
4   5. 61-72m             1407                 93            6.61   

   cumulative_churned_customers  
0                          1037  
1                          1331  
2                          1656  
3                          1776  
4                          1869   



In [74]:
#services link to higher retention
q5 = """
SELECT 'OnlineSecurity' AS service, OnlineSecurity AS status, COUNT(*) AS customers,
       ROUND(100.0 * SUM(CASE WHEN Churn = 'No' THEN 1 ELSE 0 END) / COUNT(*), 2) AS retention_rate_pct
FROM customers
GROUP BY OnlineSecurity

UNION ALL

SELECT 'TechSupport' AS service, TechSupport AS status, COUNT(*) AS customers,
       ROUND(100.0 * SUM(CASE WHEN Churn = 'No' THEN 1 ELSE 0 END) / COUNT(*), 2) AS retention_rate_pct
FROM customers
GROUP BY TechSupport

UNION ALL

SELECT 'OnlineBackup' AS service, OnlineBackup AS status, COUNT(*) AS customers,
       ROUND(100.0 * SUM(CASE WHEN Churn = 'No' THEN 1 ELSE 0 END) / COUNT(*), 2) AS retention_rate_pct
FROM customers
GROUP BY OnlineBackup

UNION ALL

SELECT 'DeviceProtection' AS service, DeviceProtection AS status, COUNT(*) AS customers,
       ROUND(100.0 * SUM(CASE WHEN Churn = 'No' THEN 1 ELSE 0 END) / COUNT(*), 2) AS retention_rate_pct
FROM customers
GROUP BY DeviceProtection
ORDER BY service, retention_rate_pct DESC;
"""
print("--- Q5: Retention Rate by Add-on Service ---")
print(q(q5), "\n")


--- Q5: Retention Rate by Add-on Service ---
             service               status  customers  retention_rate_pct
0   DeviceProtection  No internet service       1526               92.60
1   DeviceProtection                  Yes       2422               77.50
2   DeviceProtection                   No       3095               60.87
3       OnlineBackup  No internet service       1526               92.60
4       OnlineBackup                  Yes       2429               78.47
5       OnlineBackup                   No       3088               60.07
6     OnlineSecurity  No internet service       1526               92.60
7     OnlineSecurity                  Yes       2019               85.39
8     OnlineSecurity                   No       3498               58.23
9        TechSupport  No internet service       1526               92.60
10       TechSupport                  Yes       2044               84.83
11       TechSupport                   No       3473               58.36 



In [75]:
#revenue tied to lost-customer and revenue at risk
q6 = """
SELECT 
    -- Lost monthly revenue from churned customers
    ROUND(SUM(CASE WHEN Churn = 'Yes' THEN MonthlyCharges ELSE 0 END), 2) AS lost_monthly_revenue,
    ROUND(SUM(CASE WHEN Churn = 'Yes' THEN MonthlyCharges ELSE 0 END) * 12, 2) AS annualized_lost_revenue,
    
    -- Revenue-at-risk: active customers on Month-to-month contracts paying above average
    ROUND(SUM(
        CASE 
            WHEN Churn = 'No' 
             AND Contract = 'Month-to-month' 
             AND MonthlyCharges > (SELECT AVG(MonthlyCharges) FROM customers)
            THEN MonthlyCharges 
            ELSE 0 
        END
    ), 2) AS high_risk_monthly_revenue
FROM customers;
"""
print("--- Q6: Lost Revenue & Revenue-at-Risk ---")
print(q(q6), "\n")


--- Q6: Lost Revenue & Revenue-at-Risk ---
   lost_monthly_revenue  annualized_lost_revenue  high_risk_monthly_revenue
0             139130.85                1669570.2                   95178.95 



In [76]:
# INNER JOIN: Combines customers with contract duration metadata
q_inner = """
SELECT 
    ci.Contract,
    ci.contract_months,
    ci.commitment_level,
    COUNT(c.customerID) AS total_customers,
    ROUND(100.0 * SUM(CASE WHEN c.Churn = 'Yes' THEN 1 ELSE 0 END) / COUNT(*), 2) AS churn_rate_pct
FROM customers c
INNER JOIN contract_info ci 
    ON c.Contract = ci.Contract
GROUP BY ci.Contract, ci.contract_months, ci.commitment_level
ORDER BY ci.contract_months;
"""
print("--- INNER JOIN Result ---")
print(q(q_inner), "\n")

# LEFT JOIN: Keeps all customers even if a contract type wasn't in the lookup table
q_left = """
SELECT 
    c.customerID,
    c.Contract,
    ci.contract_months,
    c.MonthlyCharges,
    c.Churn
FROM customers c
LEFT JOIN contract_info ci 
    ON c.Contract = ci.Contract
LIMIT 5;
"""
print("--- LEFT JOIN Sample ---")
print(q(q_left), "\n")


--- INNER JOIN Result ---
         Contract contract_months commitment_level  total_customers  \
0  Month-to-month               1         Flexible             3875   
1        One year              12           Medium             1473   
2        Two year              24        Long-term             1695   

   churn_rate_pct  
0           42.71  
1           11.27  
2            2.83   

--- LEFT JOIN Sample ---
   customerID        Contract contract_months  MonthlyCharges Churn
0  7590-VHVEG  Month-to-month               1           29.85    No
1  5575-GNVDE        One year              12           56.95    No
2  3668-QPYBK  Month-to-month               1           53.85   Yes
3  7795-CFOCW        One year              12           42.30    No
4  9237-HQITU  Month-to-month               1           70.70   Yes 



In [77]:
pandas_churn_pct = round((df["Churn"] == "Yes").mean() * 100, 2)
sql_churn_pct = q(q1)["churn_rate_pct"].iloc[0]

assert pandas_churn_pct == sql_churn_pct, f"Mismatch: {pandas_churn_pct} vs {sql_churn_pct}"
print(f"VERIFIED: Pandas churn ({pandas_churn_pct}%) == SQL churn ({sql_churn_pct}%)")


VERIFIED: Pandas churn (26.54%) == SQL churn (26.54%)
